In [1]:
# ============================================================
# Notebook : 12_threshold_sensitivity.ipynb
# Purpose  : Sensitivity of the oversight-tier thresholds (review 3.6 / 4).
#   Under the revised HITL reading (tier 1 = automated monitoring & logging,
#   tier 2 = expert review, tier 3 = protective hold / model not used), a
#   category reaches the lightest tier only if it is fully reproduced AND its
#   precision clears a cutoff p*. This notebook sweeps p* (and the AUC cutoff)
#   to show (i) how many categories reach tier 1 at each cutoff and (ii) the
#   per-category cross-over precision, so the chosen 0.4 cutoff is shown to be
#   a deliberate, inspectable choice rather than an arbitrary one.
# Design   : Single code cell. Grayscale figures, 600 dpi, png+pdf. No path print.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.family": "serif", "axes.facecolor": "white",
                     "figure.facecolor": "white", "savefig.facecolor": "white",
                     "savefig.dpi": 600, "figure.dpi": 120,
                     "axes.edgecolor": "0.15", "grid.color": "0.85"})


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TAB = ROOT / "results" / "tables"
perf = pd.read_csv(TAB / "table_model_performance.csv")
perf.columns = [c.strip() for c in perf.columns]
CAT_KO = {"HR": "심박이상", "BR": "호흡이상", "ST": "체온이상",
          "SP": "산소포화도이상", "ER": "응급직접신호"}

# In the full home sample all five categories are fully reproduced (notebook 03),
# except ER which is a raw direct-signal category (always tier 3). The vital
# categories therefore depend on the precision cutoff alone.
reproduced = {"HR": True, "BR": True, "ST": True, "SP": True, "ER": False}
raw_direct = {"ER"}

grid = np.round(np.arange(0.10, 0.701, 0.05), 2)
counts = []
for p_star in grid:
    n_tier1 = 0
    for _, row in perf.iterrows():
        c = row["category"]
        if c in raw_direct:
            continue
        if reproduced.get(c, False) and row["precision"] >= p_star:
            n_tier1 += 1
    counts.append(n_tier1)
sweep = pd.DataFrame({"precision_cutoff": grid, "n_categories_tier1": counts})
sweep.to_csv(TAB / "table_threshold_sensitivity.csv", index=False, encoding="utf-8-sig")

# Per-category cross-over precision (the category's own precision = the cutoff
# at which it would drop out of tier 1).
cross = perf[~perf["category"].isin(raw_direct)][["category", "precision", "auc"]].copy()
cross["category_ko"] = cross["category"].map(CAT_KO)
cross["tier1_at_0.4"] = cross["precision"] >= 0.4
cross = cross.round(4)
cross.to_csv(TAB / "table_threshold_crossover.csv", index=False, encoding="utf-8-sig")

# -------- Figure: number of tier-1 categories vs precision cutoff ----------
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.step(sweep["precision_cutoff"], sweep["n_categories_tier1"], where="mid",
        color="0.15", lw=1.8)
ax.axvline(0.4, color="0.55", ls="--", lw=1.2)
ax.text(0.405, 0.3, "chosen cutoff 0.4", rotation=90, va="bottom", fontsize=8, color="0.35")
for _, r in cross.iterrows():
    ax.plot(r["precision"], 0.15, "v", color="0.35", markersize=7)
    ax.text(r["precision"], 0.28, r["category"], ha="center", fontsize=7, color="0.3")
ax.set_xlabel("precision cutoff p* for tier-1 (automated monitoring)")
ax.set_ylabel("number of categories reaching tier 1")
ax.set_ylim(0, perf.shape[0])
ax.set_yticks(range(0, perf.shape[0] + 1))
fig.tight_layout()
save_fig(fig, "fig_threshold_sensitivity")
plt.close(fig)

print("=== Tier-1 count vs precision cutoff ===")
print(sweep.to_string(index=False))
print("\n=== Per-category crossover precision ===")
print(cross.to_string(index=False))
print("\nSaved: table_threshold_sensitivity.csv, table_threshold_crossover.csv, fig_threshold_sensitivity")


=== Tier-1 count vs precision cutoff ===
 precision_cutoff  n_categories_tier1
             0.10                   4
             0.15                   4
             0.20                   4
             0.25                   4
             0.30                   4
             0.35                   4
             0.40                   3
             0.45                   3
             0.50                   3
             0.55                   3
             0.60                   3
             0.65                   2
             0.70                   1

=== Per-category crossover precision ===
category  precision    auc category_ko  tier1_at_0.4
      HR     0.7603 0.9767        심박이상          True
      BR     0.6375 0.9811        호흡이상          True
      ST     0.6621 0.9910        체온이상          True
      SP     0.3766 0.9973     산소포화도이상         False

Saved: table_threshold_sensitivity.csv, table_threshold_crossover.csv, fig_threshold_sensitivity
